In [27]:
import torch
import torch.nn as nn
import pandas as pd

In [28]:
class BiLSTM(nn.Module):
    def __init__(self, fasttext_vectors, hidden_dim, output_dim, n_layers=1, dropout=0.5):
        super().__init__()
        _, embedding_dim = fasttext_vectors.shape
        self.embedding = nn.Embedding.from_pretrained(fasttext_vectors, freeze=True)
        self.LSTM = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedd = self.embedding(text)
        _, (hidden, _) = self.LSTM(embedd)
        # hidden layer [forward_layer_1, backward_layer_1, forward_layer_2..]
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]
        hidden_cat = torch.cat((hidden_forward, hidden_backward), axis=1)
        dense_op = self.fc(self.dropout(hidden_cat))
        return self.sigmoid(dense_op)

In [29]:
# Demo cell removed: model is initialized after building vocab/embeddings.

In [30]:
!pip install gensim
from gensim.models import FastText

In [38]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import string

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
CSV_PATH = 'SMSSpamCollection'  # Your file
BATCH_SIZE = 32
MAX_VOCAB_SIZE = 25000      # Cap vocab to avoid massive memory usage

# ==========================================
# 2. BUILDING THE VOCABULARY
# ==========================================
class Vocabulary:
    def __init__(self, freq_threshold=1):
        # Index 0 for padding, 1 for unknown words
        self.itos = {0: "<PAD>", 1: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<UNK>": 1}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def tokenizer_eng(self, text):
        # Simple tokenizer: lowercase & remove punctuation
        text = text.lower()
        for char in string.punctuation:
            text = text.replace(char, "")
        return text.split()

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 2 # Start after PAD and UNK
        
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence):
                frequencies[word] = frequencies.get(word, 0) + 1
                
                # Only add if it appears enough times (cleaner data)
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
                    
    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

# Load Text from CSV
df = pd.read_csv(CSV_PATH,sep='\t', names=['label', 'text'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
# Assuming columns are 'text' and 'label' (0 or 1)
texts = df['text'].tolist()
labels = df['label'].astype(int).tolist()

# Initialize and Build Vocab
vocab = Vocabulary(freq_threshold=1)
vocab.build_vocabulary(texts)

print(f"Vocabulary Built! Size: {len(vocab)}")

# ==========================================
# 3. THE BRIDGE: GENSIM -> PYTORCH MATRIX
# ==========================================
def create_embedding_matrix(pytorch_vocab, path):
    gensim_model = FastText.load(path)
    embed_dim = gensim_model.wv.vector_size
    vocab_size = len(pytorch_vocab)
    weights_matrix = torch.zeros((vocab_size, embed_dim))
    
    found_count = 0
    for i, word in pytorch_vocab.itos.items():
        if i < 2:
            continue  # Skip PAD and UNK
        
        try:
            # Check if word is in Gensim
            if word in gensim_model.wv:
                weights_matrix[i] = torch.tensor(gensim_model.wv[word])
                found_count += 1
            else:
                # Random initialization for totally unknown words
                weights_matrix[i] = torch.randn(embed_dim)
        except KeyError:
            weights_matrix[i] = torch.randn(embed_dim)
            
    print(f"Embeddings Loaded: {found_count} / {vocab_size} words found.")
    return weights_matrix

embedding_weights = create_embedding_matrix(vocab, "spam_fasttext_gensim.model")

# Initialize model with real vocab size
HIDDEN_DIM = 64
OUTPUT_DIM = 1  # Probability of being spam
model = BiLSTM(embedding_weights, HIDDEN_DIM, OUTPUT_DIM)

# ==========================================
# 4. CUSTOM DATASET CLASS
# ==========================================
class SpamDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]
        
        # Convert "Free Money" -> [45, 202, ...]
        numericalized_text = self.vocab.numericalize(text)
        
        return torch.tensor(numericalized_text), torch.tensor(label, dtype=torch.float)

# ==========================================
# 5. COLLATION (PADDING BATCHES)
# ==========================================
# This function runs every time you fetch a batch to make sure
# all sentences in that batch have the same length (using 0s)
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx
        
    def __call__(self, batch):
        # separate text and labels
        texts = [item[0] for item in batch]
        labels = [item[1] for item in batch]
        
        # Pad them! (batch_first=True makes it [Batch, Seq_Len])
        texts = pad_sequence(texts, batch_first=True, padding_value=self.pad_idx)
        texts = texts.long()
        labels = torch.tensor(labels)
        
        return texts, labels

# ==========================================
# 6. FINAL LOADERS
# ==========================================
dataset = SpamDataset(texts, labels, vocab)

# Split into Train/Test (80/20)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

# Create the DataLoaders
# This 'train_loader' is exactly what you loop over in the training loop
train_loader = DataLoader(
    dataset=train_set, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=MyCollate(pad_idx=vocab.stoi["<PAD>"])
 )

test_loader = DataLoader(
    dataset=test_set, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=MyCollate(pad_idx=vocab.stoi["<PAD>"])
 )

print("Data Pipeline Complete. Ready to Train.")

Vocabulary Built! Size: 9663
Embeddings Loaded: 9661 / 9663 words found.
Data Pipeline Complete. Ready to Train.


In [33]:
import torch.optim as optim

In [34]:
Epochs=20
lr=1e-4
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=lr)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


In [35]:
from tqdm import tqdm

In [36]:
def evaluate(model, loader, criterion):
    model.eval()
    losses, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            losses += loss.item()
            
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return losses / len(loader), correct / total

In [39]:
for epoch in range(Epochs):
        model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            
            # Standard Forward Pass (FP32)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward Pass
            loss.backward()
            
            # --- GRADIENT CLIPPING ---
            # This prevents the 0.5 stagnation by keeping gradients in a healthy range
            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        # Validation
        val_loss, val_acc = evaluate(model, test_loader, criterion)
        # scheduler.step(val_loss)
        print(f"Epoch {epoch+1} Summary: Val Loss {val_loss:.4f}, Val Acc {val_acc:.4f}")

Epoch 1: 100%|██████████| 140/140 [00:02<00:00, 46.84it/s, loss=0.706]


Epoch 1 Summary: Val Loss 0.6899, Val Acc 0.7614


Epoch 2: 100%|██████████| 140/140 [00:03<00:00, 44.98it/s, loss=0.715]


Epoch 2 Summary: Val Loss 0.6899, Val Acc 0.7614


Epoch 3: 100%|██████████| 140/140 [00:03<00:00, 41.46it/s, loss=0.674]


...

Epoch 18 Summary: Val Loss 0.6899, Val Acc 0.7614


Epoch 19: 100%|██████████| 140/140 [00:02<00:00, 48.44it/s, loss=0.692]


Epoch 19 Summary: Val Loss 0.6899, Val Acc 0.7614


Epoch 20: 100%|██████████| 140/140 [00:03<00:00, 42.78it/s, loss=0.696]


Epoch 20 Summary: Val Loss 0.6899, Val Acc 0.7614
